In [1]:
import psycopg2
import pandas as pd
import re
import os

from pathlib import Path
from dotenv import load_dotenv

In [2]:
load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("PORT")
database = os.getenv("DB_NAME")

# Creating Database

In [3]:
try:
    conn: psycopg2.connect = psycopg2.connect(
        dbname="postgres",
        user=user,
        password=password,
        host=host,
        port=port
    )
    print("Database connected successfully.")
except:
    print("Database failed to connect.")

#Makes every SQL statement execute immediately without needing conn.commit().
conn.autocommit = True
cursor = conn.cursor()

Database connected successfully.


In [4]:
cursor.execute("SELECT 1 FROM pg_database WHERE datname = 'cyber_dashboard'")
exists = cursor.fetchone()

if not exists:
    cursor.execute("CREATE DATABASE cyber_dashboard")
    print("New Database Created.")
cursor.close()
conn.close()

In [5]:

def connect_to_db() -> psycopg2.connect:
    """
    Connects to and creates Postgre Database
    """
    try:
        conn: psycopg2.connect = psycopg2.connect(
            dbname="postgres",
            user=user,
            password=password,
            host=host,
            port=port
        )
        print("Database connected successfully.")
    except:
        print("Database failed to connect.")

    #Makes every SQL statement execute immediately without needing conn.commit().
    conn.autocommit = True
    cursor = conn.cursor()

    cursor.execute("SELECT 1 FROM pg_database WHERE datname = 'cyber_dashboard'")
    exists = cursor.fetchone()

    if not exists:
        cursor.execute("CREATE DATABASE cyber_dashboard")
        print("New Database Created.")
    cursor.close()
    conn.close()

    try:
        conn: psycopg2.connect = psycopg2.connect(
            dbname=database,
            user=user,
            password=password,
            host=host,
            port=port
        )
        print("Database connected successfully.")
    except:
        print("Database failed to connect.")

    conn.autocommit = True
    cursor = conn.cursor()
    return conn

In [ ]:
cursor.execute("CREATE TABLE testing(" \
"id SERIAL PRIMARY KEY, " \
"name TEXT)   ")

In [ ]:
cursor.execute("SELECT to_regclass('public.testing') IS NOT NULL AS table_exists")
exists = cursor.fetchone()
print(exists)

(True,)


In [ ]:
def ingest_data():
    #Connect to database
    e = connect_to_db()
    parent: str = (Path.cwd()).parent
    path: Path = Path(f"{parent}/data/")
    day_regex: str = r"[\w\-.]+(?=-[Ww]orkingHours)"
    type_regex: str = r"(?<=-[Ww]orkingHours-)[\w\-.]+(?=.pcap)"



    for file_name in path.iterdir():
        with open(file_name, mode='r') as file:
            datafile = pd.read_csv(file)
            day_match: re.Match = re.search(day_regex, str(file_name.name))
            type_match: re.Match = re.search(type_regex, str(file_name))
            if day_match:
                day_match: str = day_match.group().replace("_", " ")
            if type_match:
                type_match: str = type_match.group().replace("_", " ")
            else:
                type_match = "Working_Hours"

            database_name = f"{day_match}_{type_match}"
            
            datafile.to_sql(
                f"{database_name}",
                conn, 
                if_exists="replace",
                index=False
            )
    

    

ingest_data()

Database connected successfully.


C:\Users\emera\AppData\Local\Temp\ipykernel_18296\3921330421.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  datafile.to_sql(


SyntaxError: syntax error at or near ";"
LINE 8:             AND name=?;
                              ^
